In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy import stats
from yellowbrick.cluster import KElbowVisualizer



In [ ]:
# reading the csv file 
df = pd.read_csv("C:/Users/andsa/Documents/ML/MLing/dataset/sensor.csv")

df_subset = df.iloc[:2000, 2:12]  # Første 2000 rader, kolonne 2 til 11 (de 10 første ekskludert de to første)

# print('There are {} rows and {} columns in our dataset.'.format(df_subset.shape[0],df_subset.shape[1]))

In [ ]:
# df.head()
# df.describe()

print(df_subset.isnull().sum())  # Count NaN values per column
# df.fillna(df_subset.mean(), inplace=True)  # Fill NaNs with column means

In [ ]:
X_numerics = df_subset

model = KMeans(random_state=1)
visualizer = KElbowVisualizer(model, k=(2,10))

# visualizer.fit(X_numerics)
# visualizer.show()
# plt.show()


In [ ]:
# Select only numerical columns for clustering
df_numeric = df_subset.select_dtypes(include=[np.number])

# Standardize the data
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_numeric)

In [ ]:
KM_5_clusters = KMeans(n_clusters=5, init='k-means++').fit(X_numerics) # initialise and fit K-Means model

KM5_clustered = X_numerics.copy()
KM5_clustered.loc[:,'Cluster'] = KM_5_clusters.labels_

In [ ]:
inertia = []  # Sum of squared distances to closest cluster center
K_range = range(1, 11)  # Testing for k from 1 to 10

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    inertia.append(kmeans.inertia_)

# Plot elbow curve
plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia, marker='o', linestyle='-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.show()

In [ ]:
k_optimal = 3  # Change this based on the elbow method
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df_subset['Cluster'] = kmeans.fit_predict(df_scaled)  # Assign cluster labels


In [ ]:
plt.scatter(df_scaled[:, 0], df_scaled[:, 1], c=df_subset['Cluster'], cmap='viridis', alpha=0.6)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], c='red', marker='X', s=200, label='Centroids')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('K-Means Clustering')
plt.legend()
plt.show()

In [ ]:
print(df_subset.groupby('Cluster').mean())  # Mean values of each cluster